In [1]:
import pandas as pd
import torch
import torch.nn as nn
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
from torch.utils.data import Dataset
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence
from sklearn.model_selection import train_test_split
from torch.utils.data import Subset
import random
import pickle

In [2]:
def set_seed(seed):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False
torch.use_deterministic_algorithms(True)

set_seed(SEED)

device = torch.device("cuda")

In [3]:
class MusicDataset(Dataset):
    def __init__(self):
        with open("pakiet/train.pkl", "rb") as file:
            train_raw = pickle.load(file)
        self.xx = []
        self.yy = []
        for x, y in train_raw:
            self.xx.append(torch.tensor(x, dtype=torch.float32))
            self.yy.append(torch.tensor(y, dtype=torch.long).unsqueeze(0))

    def __len__(self):
        return len(self.xx)

    def __getitem__(self, idx):
        return self.xx[idx], self.yy[idx]

In [4]:
pad_value = 0
def pad_collate(batch):
    xx, yy = zip(*batch)
    yy = torch.stack(yy)
    x_lens = [x.shape[0] for x in xx]
    y_lens = [1] * len(yy)

    xx_pad = pad_sequence(xx, batch_first=True, padding_value=pad_value)

    return xx_pad, yy, x_lens, y_lens

In [5]:
full_trainset = MusicDataset()

train_ratio = 0.8

targets = full_trainset.yy
indices = list(range(len(full_trainset)))

train_indices, val_indices = train_test_split(
    indices,
    train_size=train_ratio,
    stratify=targets,
    random_state=SEED
)

trainset = Subset(full_trainset, train_indices)
valset = Subset(full_trainset, val_indices)

# num_classes = len(trainset.dataset.classes)

# print(f"Number of classes: {num_classes}")
print(f"Trainset size: {len(trainset)}")
print(f"Validation set size: {len(valset)}")

Trainset size: 2351
Validation set size: 588


In [6]:
batch_size = 128
num_workers = 0
trainloader = torch.utils.data.DataLoader(
    trainset,
    batch_size=batch_size,
    shuffle=True,
    num_workers=num_workers,
    pin_memory=True,
    drop_last=True,
    collate_fn=pad_collate,
    )
valloader = torch.utils.data.DataLoader(
    valset,
    batch_size=batch_size,
    shuffle=False,
    num_workers=num_workers,
    pin_memory=True,
    collate_fn=pad_collate,
    )

In [7]:
class LSTM_Seq_Regressor(nn.Module):

    def __init__(self, input_size, hidden_size, num_layers, out_size):
        super().__init__()
        self.num_layers = num_layers
        self.hidden_size = hidden_size
        self.proj_size = out_size
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            proj_size=out_size,
            batch_first=True,
        )

    def init_hidden(self, batch_size):
        hidden = torch.zeros(self.num_layers, batch_size, self.proj_size)
        state = torch.zeros(self.num_layers, batch_size, self.hidden_size)
        return hidden, state

    def forward(self, x, hidden):
        all_outputs, (h_n, c_n) = self.lstm(x, hidden)
        last_out = h_n[-1]
        return last_out

In [8]:
net = LSTM_Seq_Regressor(1,200,1,5)

In [9]:
optimizer = torch.optim.Adam(net.parameters(), lr=0.001)
loss_fun = nn.CrossEntropyLoss()
net.train()
net.to(device)

# Training loop
for epoch in range(20):
    for x, targets, x_len, target_len in trainloader:
        x = x.to(device).unsqueeze(2)
        targets = targets.to(device).squeeze(-1)
        hidden, state = net.init_hidden(x.size(0))
        hidden, state = hidden.to(device), state.to(device) 
        
        x_packed = pack_padded_sequence(x, x_len, batch_first=True, enforce_sorted=False)
        preds = net(x_packed, (hidden, state))
        
        optimizer.zero_grad()
        loss = loss_fun(preds, targets)
        loss.backward()
        optimizer.step()
    if epoch % 1 == 0:
        print(f"Epoch: {epoch}, loss: {loss.item():.3}")

Epoch: 0, loss: 1.23
Epoch: 1, loss: 1.06
Epoch: 2, loss: 1.31
Epoch: 3, loss: 1.24
Epoch: 4, loss: 1.22
Epoch: 5, loss: 1.2
Epoch: 6, loss: 1.26
Epoch: 7, loss: 1.12
Epoch: 8, loss: 1.01
Epoch: 9, loss: 1.14
Epoch: 10, loss: 1.19
Epoch: 11, loss: 1.01
Epoch: 12, loss: 1.12
Epoch: 13, loss: 1.13
Epoch: 14, loss: 1.16
Epoch: 15, loss: 0.906
Epoch: 16, loss: 0.94
Epoch: 17, loss: 0.88
Epoch: 18, loss: 0.851
Epoch: 19, loss: 0.968


In [10]:
def evaluate_accuracy(model, dataloader, device):
    model.eval()  # Set network to evaluation mode
    correct_predictions = 0
    total_samples = 0
    
    with torch.no_grad():  # Turn off gradients to save memory and computations
        for x, targets, x_len, target_len in dataloader:
            x = x.to(device).unsqueeze(2)
            targets = targets.to(device).squeeze(-1).long()  # Match shape [batch_size]
            
            # Initialize hidden state with the correct batch size
            hidden, state = model.init_hidden(x.size(0))
            hidden, state = hidden.to(device), state.to(device)
            
            x_packed = pack_padded_sequence(x, x_len, batch_first=True, enforce_sorted=False)
            preds = model(x_packed, (hidden, state))
            
            # Get the index of the max logit (the predicted class)
            pred_classes = torch.argmax(preds, dim=1)
            
            # Compare predictions to targets and sum up correct ones
            correct_predictions += (pred_classes == targets).sum().item()
            total_samples += targets.size(0)
            
    accuracy = correct_predictions / total_samples
    return accuracy

In [11]:
evaluate_accuracy(net, trainloader, device)

0.6202256944444444

In [12]:

evaluate_accuracy(net, valloader, device)

0.6105442176870748